In [ ]:
# Install and Upgrade Required Libraries
%pip install --upgrade transformers torch accelerate jinja2 gradio

In [2]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# --------------------------------------------------
# 1. Load the AI Model
# --------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype
).to(device)

model.eval()

print("Model loaded successfully!")


Loading Qwen/Qwen2.5-0.5B-Instruct on cpu...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
# --------------------------------------------------
# 2. System Instruction
# --------------------------------------------------

SYSTEM_PROMPT = {
    "role": "system",
    "content": (
        "You are a concise, helpful AI assistant "
        "for NIC e-Governance demonstrations. "
        "Explain concepts clearly and provide practical "
        "examples related to digital governance and "
        "citizen services."
    )
}

In [6]:
# --------------------------------------------------
# 3. Chatbot Response Function
# --------------------------------------------------

def ask_bot(user_message, history):

    messages = [SYSTEM_PROMPT]

    # Process the latest 6 conversation exchanges
    for item in history[-12:]:

        # Handle dictionary-based chat history
        if isinstance(item, dict):

            role = item.get("role")
            content = item.get("content")

            # Extract text from list-based content
            if isinstance(content, list):

                text_parts = []

                for part in content:

                    if isinstance(part, str):
                        text_parts.append(part)

                    elif isinstance(part, dict):
                        if part.get("type") == "text":
                            text_parts.append(
                                part.get("text", "")
                            )

                content = " ".join(text_parts)

            # Include valid messages only
            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        # Handle tuple-based history, if supplied
        elif isinstance(item, (list, tuple)) and len(item) == 2:

            user_text, assistant_text = item

            if isinstance(user_text, str):
                messages.append({
                    "role": "user",
                    "content": user_text
                })

            if isinstance(assistant_text, str):
                messages.append({
                    "role": "assistant",
                    "content": assistant_text
                })

    # Add current user message
    messages.append({
        "role": "user",
        "content": str(user_message)
    })

    # Convert messages into model prompt
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(device)

    # Generate AI response
    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract newly generated tokens
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return reply


print("Chatbot ready!")


Chatbot ready!


In [11]:
# --------------------------------------------------
# 4. Create Gradio Chat Interface
# --------------------------------------------------
demo = gr.ChatInterface(
    fn=ask_bot,

    title="NIC e-Governance AI Assistant",

    description="""
    Welcome to the NIC e-Governance AI Assistant!

    This AI-powered chatbot uses the Qwen2.5-0.5B-Instruct
    language model to answer questions about e-Governance,
    digital public services, and the role of the National
    Informatics Centre (NIC).

    **How to use:**
    1. Enter your question in the message box.
    2. Press Enter to submit your question.
    3. Read the AI-generated response.
    4. Continue the conversation with follow-up questions.

    Try one of the sample questions below to get started!
    """,

    examples=[
        "What is e-Governance?",
        "Explain the role of NIC in digital governance.",
        "How can AI improve citizen services?"
    ],

   

   chatbot=gr.Chatbot(
        height=450,
        placeholder="Hello! How can I help you with e-Governance today?"
    ),

    textbox=gr.Textbox(
        placeholder="Type your question here...",
        lines=2, 
        submit_btn="Send"
    )
)

In [12]:
# --------------------------------------------------
# 5. Launch the Application
# --------------------------------------------------

print("Launching Gradio Chatbot...")

demo.launch(
    share=True,
    inbrowser=True
)


Launching Gradio Chatbot...
* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://bd775eb527061443ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
